# Switch to a local model

In this notebook, we will:
1. Start a local model with Ollama.
2. Classify the same 10 reviews with the local model and the API model.
3. Compare accuracy, speed, cost and privacy.

**Data:** uses `data/processed/silver_labels.csv`, created once with [classify_reviews.ipynb](../03_Classify_Reviews/classify_reviews.ipynb). Reuse the file; no new data preparation is needed.


## Libraries and settings

In [ ]:
# Libraries
import os
import subprocess
import time
import platform
from datetime import datetime
from pathlib import Path
from platform import python_version

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.metrics import accuracy_score

# Folder with the data (relative to the folder of this notebook)
DATA = Path("../data")

# API model and course price assumptions in USD per 1 million tokens
MODEL = "gpt-6-luna"
PRICE_INPUT_PER_M = 0.10
PRICE_OUTPUT_PER_M = 0.50

# Local model: small enough for a codespace without a graphics card
LOCAL_MODEL = "qwen3:1.7b"

# The API key is a Codespaces secret; outside Codespaces it is read from .env
load_dotenv("../.env")

# The client sends our requests and retries when the rate limit is reached
client = OpenAI(max_retries=8)

# Show current working directory
print(os.getcwd())

## Step 1: Start Ollama and download the model

The cell has three steps: install Ollama if it is missing (in the codespace it is already installed), start the Ollama server, and download the model (about 1.4 GB, only the first time). Lines that start with `!` are shell commands.

In [ ]:
# 1. Install Ollama if it is missing (about 2 minutes)
!which ollama > /dev/null || curl -fsSL https://ollama.com/install.sh | sh

# 2. Start the Ollama server in the background.
# If the server is already running, this does nothing.
subprocess.Popen(
    ["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(3)

# 3. Download the model (only the first time) and list the installed models
!ollama pull {LOCAL_MODEL} 2>&1 | tail -n 1
!ollama list

### Connect the client

Ollama listens on this computer at `localhost:11434`. We use the same OpenAI library as before with a local address (`base_url`) and model name. Ollama ignores the API key, but the library wants one.

In [ ]:
# A client that talks to Ollama on this computer
local = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")


## Step 2: Classify 10 reviews with both models

We use 10 reviews (5 per chain) with silver labels from notebook 03. Both models get the same prompt.

In [ ]:
# Reviews with silver labels from notebook 03
silver_path = DATA / "processed/silver_labels.csv"
silver = pd.read_csv(silver_path)

# 5 random reviews per chain, the same inputs for both models
test = silver.groupby("chain").sample(n=5, random_state=42)
test = test.reset_index(drop=True)

# The same classification prompt for both models
PROMPT = (
    "Classify the sentiment of this customer review. "
    "Answer with exactly one word: positive, negative or mixed.\n"
    "\n"
    "Review: "
)

test[["id", "chain", "stars", "silver", "text"]].head(5)

### Local model

We clean each answer before we compare it with the silver label: remove spaces at both ends, use lower case and remove a final period. An answer that is then not exactly one of the three labels counts as an error.

In [ ]:
# Local model: one review after the other, and we stop the time
labels = []
start = time.perf_counter()

for text in test["text"]:
    r = local.chat.completions.create(
        model=LOCAL_MODEL,
        messages=[{"role": "user", "content": PROMPT + text}],
        reasoning_effort="none",
    )

    # Clean the answer: no spaces around it, lower case, no final period
    answer = r.choices[0].message.content
    answer = answer.strip().lower()
    labels.append(answer.rstrip("."))

local_seconds = (time.perf_counter() - start) / len(test)
test["local"] = labels

print("Seconds per review:", round(local_seconds, 2))

### API model

The same reviews go to the API model. `store=False` asks the provider not to keep the request. **Every run of this cell sends new requests and incurs API charges.** Their size depends on the texts and generated output tokens. The calculation below estimates them using the course prices.

In [ ]:
# API model: the same reviews, and we add up the tokens
labels = []
input_tokens = 0
output_tokens = 0
start = time.perf_counter()

for text in test["text"]:
    r = client.responses.create(
        model=MODEL,
        input=PROMPT + text,
        reasoning={"effort": "low"},
        store=False,
    )

    # Clean the answer in the same way as for the local model
    answer = r.output_text.strip().lower()
    labels.append(answer.rstrip("."))

    input_tokens += r.usage.input_tokens
    output_tokens += r.usage.output_tokens

api_seconds = (time.perf_counter() - start) / len(test)
test["api"] = labels

# Cost: tokens / 1 million x price per 1 million tokens
api_cost = (
    input_tokens * PRICE_INPUT_PER_M + output_tokens * PRICE_OUTPUT_PER_M
) / 1_000_000

print("Seconds per review:", round(api_seconds, 2))
print("Input tokens:", input_tokens, "| output tokens:", output_tokens)

## Step 3: Compare accuracy, speed and cost

In [ ]:
# Accuracy against the silver labels, time per review and API cost
comparison = pd.DataFrame(
    {
        LOCAL_MODEL: [
            accuracy_score(test["silver"], test["local"]),
            local_seconds,
            0.0,
        ],
        MODEL: [
            accuracy_score(test["silver"], test["api"]),
            api_seconds,
            api_cost,
        ],
    },
    index=[
        "accuracy (silver labels)",
        "seconds per review",
        f"API cost in USD ({len(test)} reviews)",
    ],
)
comparison.round(4)

In [ ]:
# Reviews where at least one model disagrees with the silver label
pd.set_option("display.max_colwidth", 300)
errors = test[
    (test["local"] != test["silver"]) | (test["api"] != test["silver"])
]
errors[["id", "silver", "local", "api", "text"]]

## Interpret the comparison

- **Accuracy:** read the measured scores and inspect the disagreements below the table. Which model makes more errors in this sample? Do not assume the winner from the model size.
- **Fair comparison?** The silver labels were made by `gpt-6-luna` itself. A model agrees with its own labels more easily; a fair comparison needs gold labels made by people.
- **10 reviews are few:** one additional correct or incorrect prediction changes the accuracy by 10 percentage points. For a real decision you would test many more.
- **Cost:** the local model has no price per token, but it is not free: it needs hardware, electricity and somebody who runs it.
- **Privacy:** with a local model, the reviews never leave the computer. That can be the deciding argument.

## Check your understanding

Answer briefly using the outputs above. Try first, then open the hints.

1. What changes when the client points to localhost instead of the external API? Where are the review texts processed?
2. Use your comparison table: which model is more accurate and which is faster? Can these results establish that one model is always better?
3. Why can silver labels from the API model favour that model in the comparison? What would make the reference more independent?
4. Why do the measured seconds only describe this computer at this moment? Why do zero API costs not mean that a local model is free to operate?
5. Give one requirement favouring local processing and one favouring an API. Before choosing either, which simpler baseline could you test for this task?

**Your answers:**

1. Was ändert sich, wenn der Client auf `localhost` statt auf die externe API zugreift?
   - Bei `localhost` werden die Bewertungen vom lokalen Modell über Ollama auf dem eigenen Computer verarbeitet. Bei der externen API werden sie an den Anbieter gesendet und dort verarbeitet
   - So erkennst du es: `base_url="http://localhost:11434/v1"` verbindet den Client mit dem lokalen Ollama-Server

2. Welches Modell ist genauer und welches schneller?
   - Die Genauigkeit und Geschwindigkeit werden zwischen dem lokalen Modell `qwen3:1.7b` und dem API-Modell verglichen. Welches besser abschneidet, hängt von den gemessenen Ergebnissen ab. Mit nur 10 Bewertungen lässt sich nicht allgemein feststellen, welches Modell besser ist
   - So erkennst du es: `comparison` zeigt Accuracy und Sekunden pro Bewertung. Im hochgeladenen Notebook sind keine ausgeführten Vergleichswerte gespeichert, deshalb lässt sich der Gewinner nicht bestimmen

3. Warum können Silver Labels das API-Modell bevorzugen?
   - Die Silver Labels wurden zuvor vom API-Modell selbst erstellt. Deshalb könnte dieses Modell seine eigenen Bewertungen leichter reproduzieren. Unabhängige, von Menschen überprüfte Gold Labels wären ein fairerer Vergleichsmassstab
   - So erkennst du es: `accuracy_score()` vergleicht beide Modelle mit `test["silver"]`, obwohl diese Referenzlabels vom API-Modell stammen

4. Warum sind die gemessenen Zeiten nicht allgemeingültig und warum ist das lokale Modell nicht kostenlos?
   - Die Geschwindigkeit hängt von Hardware, Netzwerklatenz und aktueller Auslastung ab. Ein lokales Modell verursacht zwar keine API-Gebühren, benötigt aber Hardware, Strom und Wartung
   - So erkennst du es: `local_seconds` und `api_seconds` messen die Zeit während des aktuellen Tests. Die API-Kosten des lokalen Modells werden mit `0.0` angegeben

5. Wann ist ein lokales Modell besser und wann eine API? Welche einfache Alternative könnte man testen?
   - Ein lokales Modell eignet sich, wenn die Bewertungen den eigenen Computer nicht verlassen dürfen. Eine API eignet sich, wenn wenig eigene Rechenleistung verfügbar ist. Vorher könnte man einen einfacheren Textklassifikator mit TF-IDF testen
   - So erkennst du es: Das lokale Modell verarbeitet die Texte auf dem eigenen Computer, während die API externe Rechenleistung nutzt. TF-IDF wandelt Texte anhand ihrer Wörter in Zahlenmerkmale um, die ein einfacher Klassifikator zur Stimmungsvorhersage verwenden kann


<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Hints for self-check</summary>

1. The base URL and model change. Ollama serves the local model on this computer; the external API processes the submitted text at the provider.

2. Use the measured values, even if they differ from expectations. Ten reviews and one machine are weak evidence for a general claim.

3. The API model may reproduce its own reference labels more easily. Independently created human gold labels provide a fairer reference.

4. The time depends on the hardware, the network and the load at the moment of the run. Local operation still needs hardware, electricity and maintenance.

5. For example: a requirement to keep texts on the machine favours local processing; limited hardware or low measured latency may favour an API. Test TF-IDF with a classifier as a baseline.

</details>

## Optional: test a larger sample

If you have time, change `sample(n=5, ...)` to `sample(n=15, ...)` per chain (if enough valid silver labels are available) in step 2 and rerun the comparison. Are the same conclusions supported? This takes longer and sends new API requests. Other extensions are unnecessary for the core comparison.

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Show operating system, date/time and Python version
print("-----------------------------------")
print(os.name.upper())
print(platform.system(), "|", platform.release())
print("Datetime:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("Python Version:", python_version())
print("-----------------------------------")
